#  Physics OOD Feature Test

## Objective

This notebook tests a small set of additional physics-inspired features before changing the validated forward pipeline.

The goal is **not** to replace the current baseline automatically.  
The goal is to compare:

```text
Baseline features
vs
OOD-enhanced physics features
```

Main additions tested:

- `froude_proxy`
- `sqrt_horizontal_E_over_g`
- `log_E_over_g`
- `porosity_regime_soft`
- `strength_regime_soft`

Decision rule:

```text
Adopt the new feature set only if CV improves distance targets
without materially degrading fragmentation targets.
```

This notebook does not overwrite `prediction_submission.csv` by default.

## 1. Imports and project paths

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

FORWARD_DIR = PROJECT_ROOT / "data" / "raw" / "forward_prediction"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
SUBMISSIONS_DIR = OUTPUTS_DIR / "submissions"
REPORTS_DIR = PROJECT_ROOT / "reports"

for path in [SUBMISSIONS_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Forward data:", FORWARD_DIR)
print("Reports:", REPORTS_DIR)

Project root: /home/alouiyaz/projects/boom-challenge-ejecta-prediction
Forward data: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/data/raw/forward_prediction
Reports: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/reports


## 2. Load data

In [2]:
input_cols = [
    "energy", "angle_rad", "coupling", "strength",
    "porosity", "gravity", "atmosphere", "shape_factor",
]

target_cols = [
    "P80", "fines_frac", "oversize_frac",
    "R95", "R50_fines", "R50_oversize",
]

fragmentation_targets = ["P80", "fines_frac", "oversize_frac"]
distance_targets = ["R95", "R50_fines", "R50_oversize"]

raw_train = pd.read_csv(FORWARD_DIR / "train.csv")[input_cols]
raw_test = pd.read_csv(FORWARD_DIR / "test.csv")[input_cols]
y = pd.read_csv(FORWARD_DIR / "train_labels.csv")[target_cols]

print("Raw train:", raw_train.shape)
print("Raw test:", raw_test.shape)
print("Targets:", y.shape)
display(raw_train.head())
display(y.head())

Raw train: (2930, 8)
Raw test: (492, 8)
Targets: (2930, 6)


,energy,angle_rad,coupling,strength,porosity,gravity,atmosphere,shape_factor
0,3.826405,0.818303,0.861258,1.305809,0.337215,3.71,0.781263,0.784028
1,2.828754,1.193036,0.561245,3.494501,0.058029,1.62,0.136205,0.922737
2,3.068907,0.605872,0.948860,1.366386,0.315632,3.71,0.774704,0.954922
3,2.700574,1.073708,0.713705,3.599419,0.033062,1.62,0.144204,0.932911
4,3.484022,0.863568,1.237205,1.996742,0.278207,9.81,0.414620,1.260855


,P80,fines_frac,oversize_frac,R95,R50_fines,R50_oversize
0,76.972350,0.184728,0.016671,198.938699,175.527939,76.235779
1,269.057465,0.000622,0.916734,239.268477,447.157838,141.894047
2,104.070923,0.070343,0.094438,192.986417,189.286407,84.235774
3,257.618403,0.001026,0.880122,289.289693,500.000028,169.866473
4,111.717167,0.058576,0.136166,94.229304,97.614864,42.928393


## 3. Baseline feature engineering

This is the same feature engineering used in the clean forward baseline.

In [3]:
def add_physics_features_baseline(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    eps = 1e-9

    # Energy transfer
    X["effective_energy"] = X["energy"] * X["coupling"]
    X["log_energy"] = np.log1p(X["energy"])
    X["log_effective_energy"] = np.log1p(X["effective_energy"])

    # Angle decomposition
    X["sin_angle"] = np.sin(X["angle_rad"])
    X["cos_angle"] = np.cos(X["angle_rad"])
    X["horizontal_energy"] = X["effective_energy"] * X["cos_angle"]
    X["vertical_energy"] = X["effective_energy"] * X["sin_angle"]

    # Material and fragmentation proxies
    X["energy_per_strength"] = X["energy"] / (X["strength"] + eps)
    X["effective_energy_per_strength"] = X["effective_energy"] / (X["strength"] + eps)
    X["material_resistance_index"] = X["strength"] * (1 - X["porosity"])
    X["fragmentation_index"] = X["effective_energy"] * X["porosity"] / (X["strength"] + eps)
    X["coupling_porosity"] = X["coupling"] * X["porosity"]
    X["coupling_atmosphere"] = X["coupling"] * X["atmosphere"]
    X["porosity_strength"] = X["porosity"] * X["strength"]

    # Gravity-scaled distance proxies
    X["energy_per_gravity"] = X["energy"] / (X["gravity"] + eps)
    X["effective_energy_per_gravity"] = X["effective_energy"] / (X["gravity"] + eps)
    X["horizontal_energy_per_gravity"] = X["horizontal_energy"] / (X["gravity"] + eps)
    X["vertical_energy_per_gravity"] = X["vertical_energy"] / (X["gravity"] + eps)

    # Atmosphere and drag proxies
    X["drag_proxy"] = X["atmosphere"] * X["shape_factor"]
    X["drag_per_gravity"] = X["drag_proxy"] / (X["gravity"] + eps)
    X["atmosphere_shape_energy"] = X["atmosphere"] * X["shape_factor"] * X["effective_energy"]
    X["atmosphere_per_gravity"] = X["atmosphere"] / (X["gravity"] + eps)

    # Pi-like scaling proxies
    X["pi_gravity_proxy"] = (X["gravity"] * X["coupling"]) / (X["energy"] + eps)
    X["pi_strength_proxy"] = X["strength"] / (X["gravity"] * X["coupling"] + eps)
    X["pi_atmosphere_proxy"] = X["atmosphere"] / (X["gravity"] * X["coupling"] + eps)

    # Hard regime indicators from EDA
    X["porosity_regime"] = (X["porosity"] > 0.15).astype(int)
    X["strength_regime"] = (X["strength"] > 2.6).astype(int)
    X["angle_regime"] = (X["angle_rad"] > 0.95).astype(int)
    X["atm_regime"] = (X["atmosphere"] > 0.30).astype(int)

    X["regime_combo"] = (
        X["porosity_regime"] * 8
        + X["strength_regime"] * 4
        + X["angle_regime"] * 2
        + X["atm_regime"]
    )

    # Additional interactions
    X["scaled_energy"] = X["effective_energy"] / (X["strength"] * np.sqrt(X["gravity"]) + eps)
    X["fragility"] = X["porosity"] / (X["strength"] + eps)
    X["range_proxy"] = (X["effective_energy"] * X["cos_angle"] ** 2) / (X["gravity"] * X["strength"] + eps)
    X["energy_sin_angle"] = X["energy"] * X["sin_angle"]
    X["momentum_proxy"] = X["effective_energy"] * X["sin_angle"]

    # Controlled atmosphere ratio
    X["coupling_per_atm_clipped"] = X["coupling"] / (X["atmosphere"] + 1e-3)
    X["log_coupling_per_atm"] = np.log1p(X["coupling_per_atm_clipped"])
    X["retention_factor"] = X["atmosphere"] * X["drag_proxy"] / (X["energy"] + eps)

    return X

## 4. OOD-enhanced feature engineering

This version adds a few targeted features for gravity-related extrapolation and smoother material regime transitions.

In [4]:
def add_physics_features_ood(X: pd.DataFrame) -> pd.DataFrame:
    X = add_physics_features_baseline(X)
    eps = 1e-9

    # Additional gravity-scaled proxies.
    # These are tested because test scenarios may include gravity levels not seen in training.
    X["froude_proxy"] = np.sqrt(X["effective_energy"] / (X["gravity"] + eps))
    X["sqrt_horizontal_E_over_g"] = np.sqrt(np.maximum(X["horizontal_energy"], 0) / (X["gravity"] + eps))
    X["log_E_over_g"] = np.log1p(X["effective_energy"] / (X["gravity"] + eps))

    # Soft transitions around exploratory regime thresholds.
    # This avoids a hard 0/1 jump around porosity=0.15 and strength=2.6.
    X["porosity_regime_soft"] = 1 / (1 + np.exp(-20 * (X["porosity"] - 0.15)))
    X["strength_regime_soft"] = 1 / (1 + np.exp(-5 * (X["strength"] - 2.6)))

    return X


X_base_train = add_physics_features_baseline(raw_train)
X_base_test = add_physics_features_baseline(raw_test)

X_ood_train = add_physics_features_ood(raw_train)
X_ood_test = add_physics_features_ood(raw_test)

print("Baseline train:", X_base_train.shape)
print("OOD-enhanced train:", X_ood_train.shape)

assert X_base_train.isna().sum().sum() == 0
assert X_ood_train.isna().sum().sum() == 0
assert np.isfinite(X_base_train.select_dtypes(include=[np.number]).values).all()
assert np.isfinite(X_ood_train.select_dtypes(include=[np.number]).values).all()

display(X_ood_train[[
    "gravity", "effective_energy", "horizontal_energy",
    "froude_proxy", "sqrt_horizontal_E_over_g", "log_E_over_g",
    "porosity", "porosity_regime", "porosity_regime_soft",
    "strength", "strength_regime", "strength_regime_soft"
]].head())

Baseline train: (2930, 46)
OOD-enhanced train: (2930, 51)


,gravity,effective_energy,horizontal_energy,froude_proxy,sqrt_horizontal_E_over_g,log_E_over_g,porosity,porosity_regime,porosity_regime_soft,strength,strength_regime,strength_regime_soft
0,3.71,3.295522,2.252360,0.942487,0.779170,0.635667,0.337215,1,0.976894,1.305809,0,0.001545
1,1.62,1.587625,0.585579,0.989957,0.601222,0.683105,0.058029,0,0.137121,3.494501,1,0.988710
2,3.71,2.911963,2.393651,0.885943,0.803237,0.579360,0.315632,1,0.964860,1.366386,0,0.002091
3,1.62,1.927413,0.919122,1.090762,0.753233,0.783792,0.033062,0,0.087964,3.599419,1,0.993288
4,9.81,4.310448,2.800625,0.662867,0.534310,0.364222,0.278207,1,0.928518,1.996742,0,0.046695


## 5. Feature sets

We compare the same target-specific baseline feature sets against OOD-enhanced feature sets.

In [5]:
def clean_feature_list(features, X):
    seen = set()
    out = []
    for f in features:
        if f in X.columns and f not in seen:
            out.append(f)
            seen.add(f)
    return out


raw_features = input_cols.copy()

core_physics_features = raw_features + [
    "effective_energy", "log_energy", "log_effective_energy",
    "sin_angle", "cos_angle", "horizontal_energy", "vertical_energy",
    "energy_per_strength", "effective_energy_per_strength",
    "material_resistance_index", "fragmentation_index",
    "coupling_porosity", "coupling_atmosphere", "porosity_strength",
    "energy_per_gravity", "effective_energy_per_gravity",
    "horizontal_energy_per_gravity", "vertical_energy_per_gravity",
    "drag_proxy", "drag_per_gravity", "atmosphere_shape_energy", "atmosphere_per_gravity",
]

baseline_extended_features = core_physics_features + [
    "pi_gravity_proxy", "pi_strength_proxy", "pi_atmosphere_proxy",
    "scaled_energy", "fragility", "range_proxy",
    "energy_sin_angle", "momentum_proxy",
    "coupling_per_atm_clipped", "log_coupling_per_atm", "retention_factor",
    "porosity_regime", "strength_regime", "angle_regime", "atm_regime", "regime_combo",
]

baseline_fragmentation_features = raw_features + [
    "effective_energy", "log_effective_energy",
    "sin_angle", "cos_angle", "horizontal_energy", "vertical_energy",
    "energy_per_strength", "effective_energy_per_strength",
    "material_resistance_index", "fragmentation_index",
    "coupling_porosity", "coupling_atmosphere", "porosity_strength",
    "drag_proxy", "atmosphere_shape_energy",
    "pi_strength_proxy", "pi_atmosphere_proxy", "scaled_energy", "fragility",
    "energy_sin_angle", "momentum_proxy",
    "porosity_regime", "strength_regime", "angle_regime", "atm_regime", "regime_combo",
]

ood_distance_additions = [
    "froude_proxy",
    "sqrt_horizontal_E_over_g",
    "log_E_over_g",
]

ood_fragmentation_additions = [
    "porosity_regime_soft",
    "strength_regime_soft",
]

ood_extended_features = baseline_extended_features + ood_distance_additions
ood_fragmentation_features = baseline_fragmentation_features + ood_fragmentation_additions

baseline_extended_features = clean_feature_list(baseline_extended_features, X_base_train)
baseline_fragmentation_features = clean_feature_list(baseline_fragmentation_features, X_base_train)

ood_extended_features = clean_feature_list(ood_extended_features, X_ood_train)
ood_fragmentation_features = clean_feature_list(ood_fragmentation_features, X_ood_train)

feature_summary = pd.DataFrame([
    {"feature_set": "baseline_fragmentation", "n_features": len(baseline_fragmentation_features)},
    {"feature_set": "baseline_distance", "n_features": len(baseline_extended_features)},
    {"feature_set": "ood_fragmentation", "n_features": len(ood_fragmentation_features)},
    {"feature_set": "ood_distance", "n_features": len(ood_extended_features)},
])

display(feature_summary)


def get_features(feature_version, target):
    if feature_version == "baseline":
        return baseline_fragmentation_features if target in fragmentation_targets else baseline_extended_features
    if feature_version == "ood":
        return ood_fragmentation_features if target in fragmentation_targets else ood_extended_features
    raise ValueError(feature_version)

,feature_set,n_features
0,baseline_fragmentation,34
1,baseline_distance,46
2,ood_fragmentation,36
3,ood_distance,49


## 6. Model and metrics

In [6]:
def build_extratrees_pipeline(features, random_state=42):
    selector = ColumnTransformer(
        transformers=[
            ("selected_features", "passthrough", features)
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    model = ExtraTreesRegressor(
        n_estimators=800,
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=random_state,
        n_jobs=-1,
    )

    return make_pipeline(selector, model)


def clip_predictions(preds, target):
    preds = np.asarray(preds).copy()
    if target in ["fines_frac", "oversize_frac"]:
        return np.clip(preds, 0, 1)
    return np.clip(preds, 0, None)


def regression_metrics(y_true, y_pred, target_name=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    p95_ae = np.percentile(np.abs(y_pred - y_true), 95)

    out = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "P95_AE": p95_ae,
    }

    if target_name is not None:
        out["normalized_MAE"] = mae / (y[target_name].std() + 1e-9)

    return out

## 7. Cross-validation comparison

We compare baseline vs OOD-enhanced features using the same folds and same model class.

In [7]:
def cross_validate_feature_version(feature_version, n_splits=5, random_state=42):
    X_fe = X_base_train if feature_version == "baseline" else X_ood_train

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof = pd.DataFrame(index=y.index, columns=target_cols, dtype=float)
    fold_rows = []

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X_fe), start=1):
        X_train = X_fe.iloc[train_idx]
        X_valid = X_fe.iloc[valid_idx]

        for target in target_cols:
            features = get_features(feature_version, target)

            pipe = build_extratrees_pipeline(
                features=features,
                random_state=random_state + fold,
            )

            pipe.fit(X_train, y.iloc[train_idx][target])

            pred = pipe.predict(X_valid)
            pred = clip_predictions(pred, target)

            oof.loc[valid_idx, target] = pred

            row = regression_metrics(y.iloc[valid_idx][target], pred, target_name=target)
            row.update({
                "fold": fold,
                "target": target,
                "feature_version": feature_version,
                "n_features": len(features),
            })
            fold_rows.append(row)

    overall_rows = []
    for target in target_cols:
        row = regression_metrics(y[target], oof[target], target_name=target)
        row.update({
            "target": target,
            "feature_version": feature_version,
            "n_features": len(get_features(feature_version, target)),
        })
        overall_rows.append(row)

    return pd.DataFrame(overall_rows), pd.DataFrame(fold_rows), oof


results = {}
fold_results = {}
oof_store = {}

for feature_version in ["baseline", "ood"]:
    print("Running CV for:", feature_version)

    overall_df, fold_df, oof_df = cross_validate_feature_version(
        feature_version=feature_version,
        n_splits=5,
        random_state=42,
    )

    results[feature_version] = overall_df
    fold_results[feature_version] = fold_df
    oof_store[feature_version] = oof_df

comparison = pd.concat(results.values(), ignore_index=True)

display(comparison.sort_values(["target", "normalized_MAE"]))
comparison.to_csv(REPORTS_DIR / "09b_feature_version_cv_comparison.csv", index=False)

Running CV for: baseline
Running CV for: ood


,MAE,RMSE,R2,P95_AE,normalized_MAE,target,feature_version,n_features
6,7.673455,10.195714,0.975950,21.494272,0.116697,P80,ood,36
0,7.711179,10.257217,0.975659,21.748532,0.117271,P80,baseline,34
4,50.257977,78.354081,0.899253,175.091693,0.203557,R50_fines,baseline,46
10,50.474533,78.685933,0.898397,175.549994,0.204434,R50_fines,ood,49
5,22.580781,38.485285,0.873996,81.443275,0.208239,R50_oversize,baseline,46
11,22.637834,38.635453,0.873011,80.653078,0.208765,R50_oversize,ood,49
3,40.414064,66.886848,0.921350,148.519727,0.169420,R95,baseline,46
9,40.643158,67.266730,0.920455,149.595683,0.170381,R95,ood,49
7,0.006441,0.014129,0.957569,0.031223,0.093892,fines_frac,ood,36
1,0.006487,0.014240,0.956900,0.031649,0.094558,fines_frac,baseline,34


## 8. Delta table

Negative deltas mean that the OOD-enhanced features improved the metric.

In [8]:
baseline = results["baseline"].set_index("target")
ood = results["ood"].set_index("target")

delta_rows = []

for target in target_cols:
    row = {"target": target}

    for metric in ["MAE", "RMSE", "P95_AE", "normalized_MAE"]:
        row[f"{metric}_baseline"] = baseline.loc[target, metric]
        row[f"{metric}_ood"] = ood.loc[target, metric]
        row[f"{metric}_delta_ood_minus_baseline"] = ood.loc[target, metric] - baseline.loc[target, metric]
        row[f"{metric}_pct_change"] = (
            (ood.loc[target, metric] - baseline.loc[target, metric])
            / (baseline.loc[target, metric] + 1e-9)
        )

    delta_rows.append(row)

delta_df = pd.DataFrame(delta_rows)

display(delta_df)

delta_df.to_csv(REPORTS_DIR / "09b_ood_feature_delta_vs_baseline.csv", index=False)

,target,MAE_baseline,MAE_ood,MAE_delta_ood_minus_baseline,MAE_pct_change,RMSE_baseline,RMSE_ood,RMSE_delta_ood_minus_baseline,RMSE_pct_change,P95_AE_baseline,P95_AE_ood,P95_AE_delta_ood_minus_baseline,P95_AE_pct_change,normalized_MAE_baseline,normalized_MAE_ood,normalized_MAE_delta_ood_minus_baseline,normalized_MAE_pct_change
0,P80,7.711179,7.673455,-0.037724,-0.004892,10.257217,10.195714,-0.061503,-0.005996,21.748532,21.494272,-0.254260,-0.011691,0.117271,0.116697,-0.000574,-0.004892
1,fines_frac,0.006487,0.006441,-0.000046,-0.007043,0.014240,0.014129,-0.000111,-0.007789,0.031649,0.031223,-0.000426,-0.013454,0.094558,0.093892,-0.000666,-0.007043
2,oversize_frac,0.026200,0.025958,-0.000241,-0.009208,0.035832,0.035548,-0.000284,-0.007913,0.075833,0.075663,-0.000169,-0.002233,0.072357,0.071691,-0.000666,-0.009208
3,R95,40.414064,40.643158,0.229094,0.005669,66.886848,67.266730,0.379882,0.005679,148.519727,149.595683,1.075956,0.007245,0.169420,0.170381,0.000960,0.005669
4,R50_fines,50.257977,50.474533,0.216555,0.004309,78.354081,78.685933,0.331852,0.004235,175.091693,175.549994,0.458302,0.002617,0.203557,0.204434,0.000877,0.004309
5,R50_oversize,22.580781,22.637834,0.057053,0.002527,38.485285,38.635453,0.150168,0.003902,81.443275,80.653078,-0.790197,-0.009702,0.208239,0.208765,0.000526,0.002527


## 9. Near-zone and constraint diagnostics

The inverse-design target zone is small.  
A global MAE improvement is useful, but we also need to check behavior near the target region.

In [9]:
p80_min, p80_max, r95_max = 96.0, 101.0, 175.0

def feasibility_mask(df_targets):
    return df_targets["P80"].between(p80_min, p80_max) & (df_targets["R95"] <= r95_max)

def near_zone_mask(df_targets):
    return df_targets["P80"].between(80, 120) & (df_targets["R95"] <= 250)

def constraint_metrics(y_true_df, y_pred_df):
    true_feasible = feasibility_mask(y_true_df)
    pred_feasible = feasibility_mask(y_pred_df)
    near_zone = near_zone_mask(y_true_df)

    tp = int((true_feasible & pred_feasible).sum())
    fp = int((~true_feasible & pred_feasible).sum())
    fn = int((true_feasible & ~pred_feasible).sum())

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    out = {
        "true_feasible_count": int(true_feasible.sum()),
        "pred_feasible_count": int(pred_feasible.sum()),
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn,
        "feasible_precision": precision,
        "feasible_recall": recall,
        "feasible_f1": f1,
        "near_zone_count": int(near_zone.sum()),
    }

    if near_zone.sum() > 0:
        out["near_zone_MAE_P80"] = mean_absolute_error(
            y_true_df.loc[near_zone, "P80"],
            y_pred_df.loc[near_zone, "P80"],
        )
        out["near_zone_MAE_R95"] = mean_absolute_error(
            y_true_df.loc[near_zone, "R95"],
            y_pred_df.loc[near_zone, "R95"],
        )
        out["near_zone_R95_bias"] = float(
            (y_pred_df.loc[near_zone, "R95"] - y_true_df.loc[near_zone, "R95"]).mean()
        )

    return out


constraint_rows = []

for feature_version in ["baseline", "ood"]:
    row = constraint_metrics(y, oof_store[feature_version])
    row["feature_version"] = feature_version
    constraint_rows.append(row)

constraint_df = pd.DataFrame(constraint_rows)

display(constraint_df)

constraint_df.to_csv(REPORTS_DIR / "09b_constraint_diagnostics.csv", index=False)

,true_feasible_count,pred_feasible_count,true_positive,false_positive,false_negative,feasible_precision,feasible_recall,feasible_f1,near_zone_count,near_zone_MAE_P80,near_zone_MAE_R95,near_zone_R95_bias,feature_version
0,35,39,13,26,22,0.333333,0.371429,0.351351,372,4.571787,25.205898,14.009297,baseline
1,35,38,13,25,22,0.342105,0.371429,0.356164,372,4.549138,25.206612,13.829253,ood


## 10. Test low-gravity diagnostic

This does not use labels. It checks whether OOD-enhanced features produce plausible distance predictions for low-gravity test scenarios.

In [10]:
def fit_final_pipelines(feature_version):
    X_fe = X_base_train if feature_version == "baseline" else X_ood_train

    pipelines = {}

    for i, target in enumerate(target_cols):
        features = get_features(feature_version, target)

        pipe = build_extratrees_pipeline(
            features=features,
            random_state=42 + i,
        )

        pipe.fit(X_fe, y[target])
        pipelines[target] = pipe

    return pipelines


def predict_with_pipelines(pipelines, feature_version, X_raw):
    X_fe = add_physics_features_baseline(X_raw) if feature_version == "baseline" else add_physics_features_ood(X_raw)

    preds = pd.DataFrame(index=X_raw.index)

    for target in target_cols:
        pred = pipelines[target].predict(X_fe)
        preds[target] = clip_predictions(pred, target)

    return preds


baseline_pipelines = fit_final_pipelines("baseline")
ood_pipelines = fit_final_pipelines("ood")

baseline_test_pred = predict_with_pipelines(baseline_pipelines, "baseline", raw_test)
ood_test_pred = predict_with_pipelines(ood_pipelines, "ood", raw_test)

low_g_mask = raw_test["gravity"] < 1.5

print("Low-gravity test points:", int(low_g_mask.sum()))

low_g_compare = pd.DataFrame({
    "scenario_id": np.arange(len(raw_test))[low_g_mask],
    "gravity": raw_test.loc[low_g_mask, "gravity"].values,
    "energy": raw_test.loc[low_g_mask, "energy"].values,
    "coupling": raw_test.loc[low_g_mask, "coupling"].values,
    "R95_baseline": baseline_test_pred.loc[low_g_mask, "R95"].values,
    "R95_ood": ood_test_pred.loc[low_g_mask, "R95"].values,
    "R95_delta_ood_minus_baseline": (
        ood_test_pred.loc[low_g_mask, "R95"].values
        - baseline_test_pred.loc[low_g_mask, "R95"].values
    ),
    "R50_fines_baseline": baseline_test_pred.loc[low_g_mask, "R50_fines"].values,
    "R50_fines_ood": ood_test_pred.loc[low_g_mask, "R50_fines"].values,
    "R50_oversize_baseline": baseline_test_pred.loc[low_g_mask, "R50_oversize"].values,
    "R50_oversize_ood": ood_test_pred.loc[low_g_mask, "R50_oversize"].values,
})

display(low_g_compare)

summary_low_g = low_g_compare[[
    "R95_baseline", "R95_ood", "R95_delta_ood_minus_baseline",
    "R50_fines_baseline", "R50_fines_ood",
    "R50_oversize_baseline", "R50_oversize_ood",
]].describe().T

display(summary_low_g)

low_g_compare.to_csv(REPORTS_DIR / "09b_low_gravity_test_prediction_comparison.csv", index=False)

Low-gravity test points: 181


,scenario_id,gravity,energy,coupling,R95_baseline,R95_ood,R95_delta_ood_minus_baseline,R50_fines_baseline,R50_fines_ood,R50_oversize_baseline,R50_oversize_ood
0,0,1.38,4.387076,1.305063,699.298144,738.686697,39.388553,693.655654,728.416085,309.926210,330.283826
1,2,1.02,4.384792,1.003443,796.315687,855.248947,58.933259,781.518035,816.946991,342.577850,367.653191
2,3,1.02,4.580126,0.501471,483.521925,505.420760,21.898836,569.837929,585.034478,261.189919,266.029472
3,5,1.38,4.638247,0.277523,346.279419,342.933191,-3.346228,471.189975,469.560386,198.392260,195.441918
4,6,1.02,4.288351,0.348025,392.903399,425.340037,32.436637,505.936490,534.083592,222.038754,234.273679
...,...,...,...,...,...,...,...,...,...,...,...
176,479,1.02,4.494015,1.676570,703.914045,751.331565,47.417520,694.093854,741.634482,309.218548,336.898645
177,481,1.02,4.422549,0.209379,362.287350,368.354357,6.067007,486.271740,487.718110,206.569304,205.528165
178,484,1.02,4.547372,0.634444,532.147692,555.938525,23.790833,586.830206,584.130604,270.046965,279.711625
179,488,1.38,4.509854,0.768267,534.792497,543.245306,8.452808,608.884181,599.131609,277.801803,279.000342


,count,mean,std,min,25%,50%,75%,max
R95_baseline,181.0,553.166098,159.572009,300.140284,445.273275,520.761965,650.558626,940.771618
R95_ood,181.0,578.776557,171.420066,284.889730,466.811805,540.285972,686.919986,967.176929
R95_delta_ood_minus_baseline,181.0,25.610459,19.779867,-15.250555,10.186862,23.787110,36.753552,83.255101
R50_fines_baseline,181.0,607.228680,95.737352,442.380962,542.466201,592.624052,667.102006,832.422946
R50_fines_ood,181.0,622.701884,105.100450,430.078291,558.294998,592.373458,697.106405,851.957724
R50_oversize_baseline,181.0,271.320893,50.692125,178.129917,244.246702,265.853303,294.841994,410.779436
R50_oversize_ood,181.0,281.624903,56.806007,165.449549,248.933395,272.662162,312.321979,426.939602


## 11. Optional: log1p target transform for R95 only

This test is isolated. It should be adopted only if it clearly improves R95 without creating unstable predictions.

In [11]:
def build_log_r95_model(features, random_state=42):
    base_regressor = build_extratrees_pipeline(features, random_state=random_state)

    return TransformedTargetRegressor(
        regressor=base_regressor,
        func=np.log1p,
        inverse_func=np.expm1,
    )


def cross_validate_r95_log_transform(feature_version="ood", n_splits=5, random_state=42):
    X_fe = X_base_train if feature_version == "baseline" else X_ood_train
    features = get_features(feature_version, "R95")

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_direct = np.zeros(len(y))
    oof_log = np.zeros(len(y))

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X_fe), start=1):
        X_train = X_fe.iloc[train_idx]
        X_valid = X_fe.iloc[valid_idx]

        y_train = y.iloc[train_idx]["R95"]

        direct_model = build_extratrees_pipeline(features, random_state=random_state + fold)
        log_model = build_log_r95_model(features, random_state=random_state + fold)

        direct_model.fit(X_train, y_train)
        log_model.fit(X_train, y_train)

        oof_direct[valid_idx] = clip_predictions(direct_model.predict(X_valid), "R95")
        oof_log[valid_idx] = clip_predictions(log_model.predict(X_valid), "R95")

    rows = []

    for name, pred in [
        ("direct_R95", oof_direct),
        ("log1p_R95", oof_log),
    ]:
        row = regression_metrics(y["R95"], pred, target_name="R95")
        row["variant"] = name
        row["feature_version"] = feature_version
        row["bias"] = float((pred - y["R95"]).mean())
        rows.append(row)

    return pd.DataFrame(rows), oof_direct, oof_log


RUN_LOG_R95_TEST = True

if RUN_LOG_R95_TEST:
    r95_log_results, r95_direct_oof, r95_log_oof = cross_validate_r95_log_transform(
        feature_version="ood",
        n_splits=5,
        random_state=42,
    )

    display(r95_log_results)
    r95_log_results.to_csv(REPORTS_DIR / "09b_r95_log_transform_test.csv", index=False)
else:
    print("R95 log-transform test skipped.")

,MAE,RMSE,R2,P95_AE,normalized_MAE,variant,feature_version,bias
0,40.643158,67.26673,0.920455,149.595683,0.170381,direct_R95,ood,-0.348935
1,40.555772,67.80441,0.919178,152.864841,0.170014,log1p_R95,ood,-5.842696


## 12. Decision checklist

Use this table to decide whether to adopt OOD-enhanced features.

Recommended decision:

- Adopt OOD features only if distance targets improve in CV.
- Be careful if R95 improves but R50 targets degrade.
- Do not adopt if the improvement is only visible on the unlabeled test set.
- Keep the current clean baseline if gains are small or unstable.

In [12]:
decision_rows = []

for target in target_cols:
    mae_delta = float(delta_df.loc[delta_df["target"] == target, "MAE_delta_ood_minus_baseline"].iloc[0])
    p95_delta = float(delta_df.loc[delta_df["target"] == target, "P95_AE_delta_ood_minus_baseline"].iloc[0])

    decision_rows.append({
        "target": target,
        "MAE_delta_ood_minus_baseline": mae_delta,
        "P95_AE_delta_ood_minus_baseline": p95_delta,
        "improved_MAE": mae_delta < 0,
        "improved_P95_AE": p95_delta < 0,
    })

decision_df = pd.DataFrame(decision_rows)

display(decision_df)

distance_improved = decision_df[decision_df["target"].isin(distance_targets)]["improved_MAE"].sum()
fragmentation_degraded = (
    decision_df[decision_df["target"].isin(fragmentation_targets)]["MAE_delta_ood_minus_baseline"] > 0
).sum()

print("Distance targets improved in MAE:", int(distance_improved), "/ 3")
print("Fragmentation targets degraded in MAE:", int(fragmentation_degraded), "/ 3")

if distance_improved >= 2 and fragmentation_degraded <= 1:
    print("Candidate decision: OOD-enhanced features are promising. Review P95 and near-zone metrics before adopting.")
else:
    print("Candidate decision: Keep baseline unless another diagnostic strongly supports OOD-enhanced features.")

,target,MAE_delta_ood_minus_baseline,P95_AE_delta_ood_minus_baseline,improved_MAE,improved_P95_AE
0,P80,-0.037724,-0.254260,True,True
1,fines_frac,-0.000046,-0.000426,True,True
2,oversize_frac,-0.000241,-0.000169,True,True
3,R95,0.229094,1.075956,False,False
4,R50_fines,0.216555,0.458302,False,False
5,R50_oversize,0.057053,-0.790197,False,True


Distance targets improved in MAE: 0 / 3
Fragmentation targets degraded in MAE: 0 / 3
Candidate decision: Keep baseline unless another diagnostic strongly supports OOD-enhanced features.


## 13. Optional save of OOD submission candidate

This is disabled by default to avoid overwriting the official submission.

Set `CREATE_OOD_CANDIDATE_SUBMISSION = True` only after reviewing the CV comparison.

In [13]:
CREATE_OOD_CANDIDATE_SUBMISSION = False

if CREATE_OOD_CANDIDATE_SUBMISSION:
    candidate_submission = pd.DataFrame({
        "scenario_id": np.arange(len(raw_test))
    })

    for target in target_cols:
        candidate_submission[target] = ood_test_pred[target].values

    candidate_submission = candidate_submission[
        ["scenario_id", "P80", "fines_frac", "oversize_frac", "R95", "R50_fines", "R50_oversize"]
    ]

    candidate_path = SUBMISSIONS_DIR / "prediction_submission_09b_ood_features_candidate.csv"
    candidate_submission.to_csv(candidate_path, index=False)

    print("Saved OOD candidate submission:", candidate_path)
    display(candidate_submission.head())
else:
    print("Candidate submission creation disabled.")
    print("This notebook is for testing only and does not overwrite prediction_submission.csv.")

Candidate submission creation disabled.
This notebook is for testing only and does not overwrite prediction_submission.csv.


## 14. Notebook summary

This notebook compares the validated baseline feature set against a small OOD-enhanced physics feature set.

The most important outputs are:

- CV comparison table;
- delta table versus baseline;
- near-zone and constraint diagnostics;
- low-gravity test prediction check;
- optional R95 log-transform test.

Final adoption should be based on CV and plausibility, not on test-set intuition alone.